# Iterators, generators and decorators
* this is one of the more "advanced feeling" lessons, but honestly its just a couple of small ideas stacked on top of each other
* iterators and generators are about how python actually walks through data one item at a time (whats really happening behind every `for` loop)
* decorators are about wrapping a function with extra behaviour without touching its original code
* we assume you already know functions, `*args`/`**kwargs` and closures from lesson 04, and we only need one tiny bit of class syntax (`__init__`, `self`) which we explain inline as we go

## the iterator protocol
* "iterable" = something you CAN loop over (a list, a string, a dict, a range, ...) - it has an `__iter__` method that gives you back an iterator
* "iterator" = the actual thing doing the walking - it has a `__next__` method that returns the next value, and when there is nothing left it raises `StopIteration`
* a `for` loop is really just syntactic sugar, under the hood python calls `iter()` once and then `next()` again and again until `StopIteration` shows up
* lets prove that by doing it manually

In [1]:
# a normal list is iterable, but its NOT an iterator itself
my_list = [10, 20, 30]

# iter() asks the list for an iterator - this is what a "for" loop does internally, invisibly
my_iterator = iter(my_list)
print(type(my_list), "-->", type(my_iterator))  # list vs list_iterator, two different things!

# next() pulls one value at a time out of the iterator, and moves it forward permanently
print(next(my_iterator))  # 10
print(next(my_iterator))  # 20
print(next(my_iterator))  # 30

# theres nothing left now, so calling next() again raises StopIteration - this exception is how python KNOWS the loop is done
try:
    next(my_iterator)
except StopIteration:
    print("StopIteration raised - the iterator is exhausted, no more values")

# a "for" loop basically does exactly this in a loop, catching StopIteration for us automatically:
manual_iterator = iter(my_list)
while True:
    try:
        value = next(manual_iterator)  # get the next value
    except StopIteration:
        break  # nothing left, stop the loop - this is exactly what "for" does behind the scenes
    print("manual for-loop-equivalent got:", value)

<class 'list'> --> <class 'list_iterator'>
10
20
30
StopIteration raised - the iterator is exhausted, no more values
manual for-loop-equivalent got: 10
manual for-loop-equivalent got: 20
manual for-loop-equivalent got: 30


## building our own iterator class
* to make our own iterator we just need a class with `__iter__` (returns an iterator, usually just `self`) and `__next__` (returns the next value or raises `StopIteration`)
* `self` refers to "this specific instance of the class", `__init__` runs once automatically when we create a new object - thats really all the class-syntax we need here
* lets build a `Countdown` class that counts down from a start value to zero

In [2]:
class Countdown:
    def __init__(self, start):
        self.current = start  # remember where we currently are, stored on the instance (self)

    def __iter__(self):
        # an iterator needs to be iterable too (so it also works directly in a "for" loop)
        # we simply return ourselves here, because THIS object already has __next__
        return self

    def __next__(self):
        if self.current < 0:
            raise StopIteration  # we are done counting down, tell python to stop
        value = self.current  # remember the value we want to return this time
        self.current -= 1     # move our internal state one step further down, for next time
        return value

# we can use it exactly like a list in a for loop, python calls iter() and next() for us
for number in Countdown(3):
    print("countdown:", number)

# ...or we can drive it manually with iter()/next(), exactly like we did with the list above
manual_countdown = iter(Countdown(2))
print(next(manual_countdown))
print(next(manual_countdown))
print(next(manual_countdown))
# next(manual_countdown) here would now raise StopIteration, same as before

countdown: 3
countdown: 2
countdown: 1
countdown: 0
2
1
0


## generator functions - the shortcut for writing iterators
* writing a whole class with `__iter__`/`__next__` just to walk through some values is a lot of boilerplate
* a "generator function" is a normal function that uses `yield` instead of (or in addition to) `return` - python then builds the whole iterator machinery for us automatically
* lets rewrite `Countdown` as a generator function and see how much shorter it gets

In [3]:
def countdown_generator(start):
    # notice: no class, no self, no __init__/__iter__/__next__ - just a normal function with "yield"
    while start >= 0:
        yield start  # this "pauses" the function here and hands back "start" to whoever is calling next()
        start -= 1   # this line only runs once someone asks for the NEXT value

# using it looks identical to using our class-based Countdown
for number in countdown_generator(3):
    print("generator countdown:", number)

# important: CALLING a generator function does NOT run the body at all yet!
gen = countdown_generator(2)
print("we just called the function, but nothing was printed above - gen is:", gen, type(gen))
# calling it only gave us back a generator object, the code inside hasnt executed a single line so far

# every next() call resumes execution right after the last "yield" and runs until the NEXT "yield" (or the function ends)
print(next(gen))  # runs the function body up to the first yield -> 2
print(next(gen))  # resumes right after that yield, runs until the next yield -> 1
print(next(gen))  # -> 0
try:
    next(gen)  # the while loop condition is now False, the function just ends -> StopIteration, same as any other iterator
except StopIteration:
    print("generator exhausted, StopIteration raised, exactly like our class-based iterator")

generator countdown: 3
generator countdown: 2
generator countdown: 1
generator countdown: 0
we just called the function, but nothing was printed above - gen is: <generator object countdown_generator at 0x7fb310f996c0> <class 'generator'>
2
1
0
generator exhausted, StopIteration raised, exactly like our class-based iterator


## generator expressions vs list comprehensions
* a generator expression looks almost like a list comprehension, but with round brackets instead of square brackets: `(x for x in ...)` vs `[x for x in ...]`
* the key difference: a list comprehension builds the ENTIRE list in memory immediately, a generator expression computes each value lazily, one at a time, only when asked for it
* this makes generators way more memory efficient for big or even infinite sequences - lets measure it with `sys.getsizeof()`

In [4]:
import sys

# a list comprehension - this immediately creates ALL 1 million ints and stores them in memory
big_list = [x for x in range(1_000_000)]

# a generator expression - this creates a tiny generator object, no values are computed yet!
big_generator = (x for x in range(1_000_000))

print("list size in bytes:      ", sys.getsizeof(big_list))
print("generator size in bytes: ", sys.getsizeof(big_generator))
# the generator stays tiny no matter how many items it will eventually produce, the list grows with every item
# this is the whole point of "lazy evaluation" - you pay the memory cost only for what you actually consume

# we can still loop over a generator expression exactly like a list comprehension
total = sum(x for x in range(10))  # note: we dont even need extra brackets when its the only argument to a function
print("sum of generator expression:", total)

list size in bytes:       8448728
generator size in bytes:  200
sum of generator expression: 45


## infinite generators
* since a generator only computes a value when asked, we can build generators that never end - as long as we NEVER try to turn them into a list directly (that would run forever and eat all your memory!)
* `itertools.islice(iterable, n)` lets us safely grab just the first `n` values out of a (possibly infinite) iterator, without ever asking for "all of it" 

In [5]:
from itertools import islice

def infinite_counter(start=0):
    # this generator literally never stops by itself, "while True" has no exit condition
    while True:
        yield start
        start += 1

def fibonacci_generator():
    # another classic infinite generator - fibonacci numbers, forever
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b  # move both numbers one step forward, python evaluates the right side fully before reassigning

# NEVER do list(infinite_counter()) - it would try to build an infinite list and just hang / crash your memory!
# instead we use islice() to politely ask for only the first few values
first_five_counts = list(islice(infinite_counter(10), 5))
print("first 5 from infinite_counter(10):", first_five_counts)

first_ten_fibs = list(islice(fibonacci_generator(), 10))
print("first 10 fibonacci numbers:", first_ten_fibs)

first 5 from infinite_counter(10): [10, 11, 12, 13, 14]
first 10 fibonacci numbers: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


## itertools tour
* the `itertools` module is a whole standard-library toolbox full of clever, memory-efficient iterator building blocks
* here is a short, runnable example for the most commonly used ones

In [6]:
import itertools

# chain() - glues multiple iterables together into one, one after another, without copying them into a new list first
chained = list(itertools.chain([1, 2], [3, 4], [5]))
print("chain:", chained)

# count() - an infinite counter (basically what we hand-built above), with optional start/step - ALWAYS combine with islice!
counted = list(itertools.islice(itertools.count(10, 5), 4))  # start at 10, step by 5, take 4 values
print("count:", counted)

# cycle() - repeats an iterable forever, round and round - again, only ever consume it through islice, never list() it directly!
cycled = list(itertools.islice(itertools.cycle(["a", "b", "c"]), 7))  # goes a,b,c,a,b,c,a
print("cycle:", cycled)

# product() - cartesian product, every combination of one item from each input iterable (like nested for loops, but shorter)
producted = list(itertools.product([1, 2], ["x", "y"]))
print("product:", producted)

# permutations() - every possible ORDERING of a given length (order matters, no repeats of the same element)
permuted = list(itertools.permutations([1, 2, 3], 2))
print("permutations:", permuted)

# combinations() - every possible SELECTION of a given length (order does NOT matter, no repeats)
combined = list(itertools.combinations([1, 2, 3], 2))
print("combinations:", combined)

# groupby() - groups CONSECUTIVE elements that share the same key, very handy on pre-sorted/pre-grouped data
data = [("fruit", "apple"), ("fruit", "banana"), ("veg", "carrot"), ("veg", "potato")]
for key, group in itertools.groupby(data, key=lambda pair: pair[0]):
    print("groupby:", key, "-->", list(group))  # group is itself an iterator, so we wrap it in list() to actually see it

chain: [1, 2, 3, 4, 5]
count: [10, 15, 20, 25]
cycle: ['a', 'b', 'c', 'a', 'b', 'c', 'a']
product: [(1, 'x'), (1, 'y'), (2, 'x'), (2, 'y')]
permutations: [(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
combinations: [(1, 2), (1, 3), (2, 3)]
groupby: fruit --> [('fruit', 'apple'), ('fruit', 'banana')]
groupby: veg --> [('veg', 'carrot'), ('veg', 'potato')]


## decorators - first principles
* a decorator is really nothing magic: its just a function that takes ANOTHER function as input, and returns a (usually wrapping) function as output
* lets build one completely manually first, without any special syntax at all, so we see exactly whats going on

In [7]:
def my_decorator(func):
    # my_decorator takes a function "func" and returns a brand new function "wrapper" that wraps around it
    def wrapper():
        print("something happens BEFORE the original function runs")
        func()  # here we actually call the original function we were given
        print("something happens AFTER the original function runs")
    return wrapper  # we return the wrapper function ITSELF (not the result of calling it!)

def say_hi():
    print("hi!")

# the fully manual way: we call my_decorator ourselves and store its result
wrapped_say_hi = my_decorator(say_hi)
wrapped_say_hi()  # this now runs wrapper(), which runs our extra prints AROUND the original say_hi()

print("---")

# python gives us a shortcut syntax "@decorator_name" placed directly above a function definition
# this line...
@my_decorator
def say_hi_sugar():
    print("hi with sugar!")

# ...is EXACTLY equivalent to writing: say_hi_sugar = my_decorator(say_hi_sugar)
# the @ syntax is pure "syntactic sugar" - it does the exact same reassignment for us automatically
say_hi_sugar()

something happens BEFORE the original function runs
hi!
something happens AFTER the original function runs
---
something happens BEFORE the original function runs
hi with sugar!
something happens AFTER the original function runs


## a practical decorator: passthrough args and functools.wraps
* our `my_decorator` above only worked for functions with zero parameters - `*args`/`**kwargs` in the wrapper fixes that, it just forwards whatever it received straight to the original function
* theres one annoying side effect of wrapping a function: the wrapper REPLACES the original, so things like `__name__` and `__doc__` (used by `help()`, debuggers, documentation tools) suddenly point to the wrapper instead of the real function
* `functools.wraps` fixes this by copying that metadata from the original function onto the wrapper - lets see the broken version first, then the fixed one

In [8]:
import time
import functools

def timer_broken(func):
    # this version works fine functionally, but does NOT preserve the original function's identity/metadata
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)  # *args/**kwargs forward WHATEVER we were called with, to the real function
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f} seconds")
        return result
    return wrapper

@timer_broken
def slow_add_broken(a, b):
    """Add two numbers, slowly, on purpose."""
    time.sleep(0.01)
    return a + b

print(slow_add_broken(2, 3))
print("name looks wrong:", slow_add_broken.__name__)  # prints "wrapper" - not what we want! we lost the real name
print("docstring looks wrong:", slow_add_broken.__doc__)  # prints None - we lost the real docstring too

print("---")

def timer(func):
    @functools.wraps(func)  # this copies __name__, __doc__ (and a bit more) from "func" onto "wrapper" for us
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f} seconds")
        return result
    return wrapper

@timer
def slow_add(a, b):
    """Add two numbers, slowly, on purpose."""
    time.sleep(0.01)
    return a + b

print(slow_add(2, 3))
print("name looks correct now:", slow_add.__name__)  # prints "slow_add" - fixed!
print("docstring looks correct now:", slow_add.__doc__)  # prints our real docstring - fixed!

slow_add_broken took 0.010207 seconds
5
name looks wrong: wrapper
docstring looks wrong: None
---
slow_add took 0.010150 seconds
5
name looks correct now: slow_add
docstring looks correct now: Add two numbers, slowly, on purpose.


## decorator factories - decorators with their OWN arguments
* sometimes we want to configure a decorator, for example `@repeat(times=3)` to run a function multiple times
* this needs one extra level of nesting: an outer function that takes the decorator's OWN arguments, and returns the actual decorator (which then takes the function and returns the wrapper) - three levels total
* level 1: `repeat(times)` - takes the decorator's own argument
* level 2: `decorator(func)` - takes the function to wrap (this is the "real" decorator)
* level 3: `wrapper(*args, **kwargs)` - the thing that actually runs when the decorated function gets called

In [9]:
def repeat(times):
    # level 1: this is what gets called first, because of "repeat(times=3)" - it just captures "times" and returns the real decorator
    def decorator(func):
        # level 2: this IS the actual decorator now, it receives the function to wrap - just like my_decorator before
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # level 3: this runs every time the decorated function is called
            result = None
            for i in range(times):
                print(f"call {i + 1}/{times}")
                result = func(*args, **kwargs)  # we call the real function each time, forwarding all arguments
            return result  # we just return the last result here, keeping it simple
        return wrapper
    return decorator

@repeat(times=3)
def greet(name):
    print(f"hello {name}!")

greet("Alex")  # this single call actually runs the real greet() three times, because of our decorator factory
# remember: "@repeat(times=3)" is equivalent to "greet = repeat(times=3)(greet)" - repeat(times=3) returns the decorator, which then wraps greet

call 1/3
hello Alex!
call 2/3
hello Alex!
call 3/3
hello Alex!


## functools.lru_cache - memoization as a ready-made decorator
* "memoization" means: remember results of expensive function calls, so we dont have to recompute them again for the same input
* `functools.lru_cache` is a decorator from the standard library that does exactly this for us, zero manual caching code needed
* the classic example is the naive recursive fibonacci from lesson 04, which recalculates the same values over and over again

In [10]:
from functools import lru_cache
import time

def fib_naive(n):
    # naive recursive fibonacci, no caching - recomputes the same smaller fib() calls again and again
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

@lru_cache(maxsize=None)  # maxsize=None means "remember everything, no limit" - fine for our small example here
def fib_cached(n):
    # exact same logic as fib_naive, the ONLY difference is the decorator above
    if n <= 1:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)

n = 28  # kept modest on purpose so even the naive, uncached version finishes reasonably fast for this lesson

start = time.perf_counter()
result_naive = fib_naive(n)
time_naive = time.perf_counter() - start
print(f"fib_naive({n})  = {result_naive}, took {time_naive:.4f} seconds")

start = time.perf_counter()
result_cached = fib_cached(n)
time_cached = time.perf_counter() - start
print(f"fib_cached({n}) = {result_cached}, took {time_cached:.4f} seconds")

# calling it again is essentially instant, because lru_cache already remembers EVERY sub-result from the call above
start = time.perf_counter()
fib_cached(n)
time_cached_again = time.perf_counter() - start
print(f"fib_cached({n}) called AGAIN took {time_cached_again:.6f} seconds - basically a dictionary lookup now")

# lru_cache even gives us stats about how well the cache is doing
print(fib_cached.cache_info())

fib_naive(28)  = 317811, took 0.0420 seconds
fib_cached(28) = 317811, took 0.0001 seconds
fib_cached(28) called AGAIN took 0.000033 seconds - basically a dictionary lookup now
CacheInfo(hits=27, misses=29, maxsize=None, currsize=29)


## quick comparison to other languages
* python generators (`yield`) are directly comparable to JavaScript generator functions (`function*` with `yield`) and to C#'s `yield return` (backing an `IEnumerable`) - all three suspend and resume execution at the `yield` point, keeping their local state alive in between
* Java has no native `yield`-based generator syntax - the closest equivalent is writing your own `Iterator` class by hand (basically what we did with `Countdown` before switching to `yield`), or since Java 9 the more heavyweight `Stream`/`Spliterator` machinery
* Rust has iterators too, via the `Iterator` trait and its `next()` method (very close in spirit to our manual `__iter__`/`__next__` class), but stable Rust still has no `yield` keyword to write a generator function like python/JS/C# can
* python decorators (`@decorator`) are conceptually a bit similar to Java annotations or C# attributes (`@Override`, `[Obsolete]`), but theres a big difference: python decorators are actual, executable code that really wraps and can change the function, while Java/C# annotations/attributes are just passive metadata that some OTHER framework reads and reacts to later
* python decorators are actually closer in spirit to JavaScript's (still experimental/proposal-stage) decorator syntax, or to general "middleware" wrapping patterns you might know from web frameworks - a piece of code that wraps another piece of code to add behaviour around it

## Exercises
1. write a generator function `even_numbers_up_to(n)` that yields every even number from 0 up to (and including) `n`
2. write a decorator `uppercase_result` that takes a function returning a string, and makes it return the UPPERCASE version of that string instead
3. write a decorator factory `retry(times)` that calls the wrapped function up to `times` times if it raises an exception - only re-raise the exception after the very last attempt also failed
4. use `itertools.combinations` to print every unique pair (2 elements, order doesnt matter) out of `["a", "b", "c", "d"]`

In [11]:
# TODO: solve exercise 1 here (even_numbers_up_to generator)


# TODO: solve exercise 2 here (uppercase_result decorator)


# TODO: solve exercise 3 here (retry(times) decorator factory)


# TODO: solve exercise 4 here (itertools.combinations pairs)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [12]:
# sample solution 1 - even_numbers_up_to generator
def even_numbers_up_to(n):
    for number in range(0, n + 1):  # go through every number from 0 to n, inclusive
        if number % 2 == 0:         # only keep the even ones
            yield number            # yield hands back one value at a time, pausing here until the next next()

print(list(even_numbers_up_to(10)))

# sample solution 2 - uppercase_result decorator
def uppercase_result(func):
    @functools.wraps(func)  # keep the original name/docstring intact, good habit for every decorator
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)  # run the original function, forwarding all arguments
        return result.upper()           # transform its string result before handing it back
    return wrapper

@uppercase_result
def shout(name):
    return f"hello {name}"

print(shout("alex"))

# sample solution 3 - retry(times) decorator factory
def retry(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)  # try running it, if it works we return immediately
                except Exception as error:
                    print(f"attempt {attempt}/{times} failed with: {error}")
                    if attempt == times:
                        raise  # this was the last allowed attempt, let the exception bubble up for real now
        return wrapper
    return decorator

attempts_made = []

@retry(times=3)
def flaky_function():
    attempts_made.append(1)
    if len(attempts_made) < 3:
        raise ValueError("not yet!")
    return "finally worked"

print(flaky_function())

# sample solution 4 - itertools.combinations pairs
import itertools
letters = ["a", "b", "c", "d"]
for pair in itertools.combinations(letters, 2):
    print(pair)

[0, 2, 4, 6, 8, 10]
HELLO ALEX
attempt 1/3 failed with: not yet!
attempt 2/3 failed with: not yet!
finally worked
('a', 'b')
('a', 'c')
('a', 'd')
('b', 'c')
('b', 'd')
('c', 'd')


* congratulation you finished this lesson

## what we learned
* the iterator protocol - `__iter__` returns an iterator, `__next__` returns the next value and raises `StopIteration` when done, and this is exactly what a `for` loop does for us automatically
* how to build our own iterator class, and how a generator function with `yield` gives us the same behaviour with way less code
* calling a generator function returns a generator object immediately without running any code - the body only runs (and pauses again) as `next()` gets called
* generator expressions `(x for x in ...)` are lazily evaluated and memory-efficient compared to list comprehensions, which build the whole result upfront
* infinite generators are totally fine, as long as we only ever consume a limited slice of them with `itertools.islice`
* a whistle-stop tour of `itertools`: `chain`, `count`, `cycle`, `product`, `permutations`, `combinations`, `groupby`
* decorators are just functions that take a function and return a (usually wrapping) function - `@decorator` is pure syntactic sugar for `func = decorator(func)`
* `*args`/`**kwargs` let a decorator's wrapper forward any arguments, and `functools.wraps` preserves the original function's `__name__`/`__doc__`
* decorator factories (a decorator that itself takes arguments, like `@repeat(times=3)`) need one extra level of nested functions
* `functools.lru_cache` gives us memoization/caching as a one-line, ready-made decorator